# Data Transformation & Analytical Model

The raw Olist dataset consists of multiple related tables representing different business entities and transactions.

At this stage, the goal is not to combine every table into one large dataset. Instead, we will preserve the correct **data grain** of each entity and create analytical datasets only where required.

## Core Tables and Their Grain

| Table | Grain |
|---|---|
| `customers` | One customer-order record |
| `orders` | One order |
| `order_items` | One item within an order |
| `products` | One product |
| `sellers` | One seller |
| `payments` | One payment record |
| `reviews` | One review record |

## Important Relationships

- One customer can have multiple orders.
- One order can contain multiple order items.
- One product can appear in multiple order items.
- One seller can fulfill multiple order items.
- One order can have multiple payment records.
- One order can have multiple review records.

Because payments and reviews can contain multiple records per order, they should not be directly joined to the order-item table without first defining an appropriate aggregation.

## Analytical Grain

For sales and product analysis, the primary analytical grain will be:

> **One row = one order item**

For customer-level analysis, we will create customer-level aggregated datasets.

For order-level analysis, we will create order-level summaries such as payment and review metrics.

The final PostgreSQL database will preserve the relational tables, while analytical views/tables will be created on top of them for SQL analysis and Power BI.

In [2]:
import pandas as pd
import numpy as np

In [3]:
orders = pd.read_csv("../data/processed/orders_cleaned.csv")
order_items = pd.read_csv("../data/processed/order_items_cleaned.csv")
products = pd.read_csv("../data/processed/products_cleaned.csv")
customers = pd.read_csv("../data/processed/customers_cleaned.csv")
sellers = pd.read_csv("../data/processed/sellers_cleaned.csv")
payments = pd.read_csv("../data/processed/payments_cleaned.csv")
reviews = pd.read_csv("../data/processed/reviews_cleaned.csv")

In [4]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_approved_at"] = pd.to_datetime(
    orders["order_approved_at"]
)

orders["order_delivered_carrier_date"] = pd.to_datetime(
    orders["order_delivered_carrier_date"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"]
)

reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [6]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


## 1. Merge Order Items with Orders

The `order_items` table contains item-level information such as `product_id`, `seller_id`, `price`, and `freight_value`, while the `orders` table contains order-level information such as `customer_id`, `order_status`, and order timestamps.

Our goal is to create an **order-item-level analytical dataset** by adding the relevant order information to each order-item record.

The relationship between the tables is:

- One order can contain multiple order items.
- Each order item belongs to exactly one order.
- Therefore, `orders` → `order_items` is a **one-to-many relationship**.

We use a **LEFT JOIN** with `order_items` as the left table so that every order-item record is preserved:

`order_items LEFT JOIN orders ON order_id`

The expected grain of the resulting DataFrame is:

> **One row = one order item**

We will verify the merge by checking that the number of rows remains unchanged and that the number of unique orders is consistent with the order-item table.

In [7]:
order_items_orders = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

In [8]:
order_items.shape


(112650, 7)

In [9]:
order_items_orders.shape

(112650, 14)

In [10]:
order_items_orders["order_id"].nunique()

98666

#### 99,441 total orders
#### 98,666 orders appear in order_items
### 99,441 - 98,666 = 775 orders with no order-item records.

In [11]:
order_items_orders.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17


## 2. Add Product Information

The current `order_items_orders` DataFrame contains item-level transaction information along with details about the corresponding order.

The `products` table contains information about the products being sold, such as:

- `product_category_name`
- `product_name_lenght`
- `product_description_lenght`
- `product_photos_qty`
- Product weight and dimensions

We want to add this product information to each order-item record so that we can later perform analyses such as sales by product category and product-level performance.

The relationship is:

- One product can appear in many order items.
- Each order item refers to one product through `product_id`.
- Therefore, `products` → `order_items` is a **one-to-many relationship**.

We use a **LEFT JOIN** with `order_items_orders` as the left table so that every transaction remains in the analytical dataset:

`order_items_orders LEFT JOIN products ON product_id`

The expected grain remains:

> **One row = one order item**

We will verify that the number of rows and unique orders remain unchanged after the merge.

In [12]:
order_items_orders_products = order_items_orders.merge(
    products,
    on="product_id",
    how="left"
)

In [13]:
order_items_orders_products["order_id"].nunique()

98666

## 3. Add Seller Information

The current `order_items_orders_products` DataFrame contains order-item, order, and product information.

The `sellers` table contains information about the sellers responsible for fulfilling the products, such as:

- `seller_id`
- `seller_zip_code_prefix`
- `seller_city`
- `seller_state`

We want to add seller information so that we can later analyze seller performance and geographic patterns.

The relationship is:

- One seller can be associated with many order items.
- Each order item belongs to one seller through `seller_id`.
- Therefore, `sellers` → `order_items` is a **one-to-many relationship**.

We use a **LEFT JOIN** with `order_items_orders_products` as the left table:

`order_items_orders_products LEFT JOIN sellers ON seller_id`

The expected grain remains:

> **One row = one order item**

Since we previously verified that every `seller_id` appearing in `order_items` exists in the `sellers` table, the number of rows should remain unchanged.

We will validate the merge by checking the total number of rows and unique orders.

In [14]:
order_items_orders_products_sellers = order_items_orders_products.merge(
    sellers,
    on="seller_id",
    how="left"
)

In [15]:
order_items_orders_products_sellers.shape

(112650, 25)

In [16]:
order_items_orders_products_sellers["order_id"].nunique()

98666

In [17]:
order_items_orders_products_sellers.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,58.0,598.0,4.0,650.0,28.0,9.0,14.0,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,42.0,480.0,1.0,200.0,16.0,10.0,15.0,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,87900,loanda,PR


In [18]:
order_items_orders_products_sellers["seller_id"].isna().sum()

np.int64(0)

In [20]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


## 4. Load Cleaned Data into PostgreSQL

The cleaned datasets are stored locally in `data/processed/` and the relational PostgreSQL schema has been created in Supabase.

The next step is to load the processed data into the corresponding PostgreSQL tables.

The database will preserve the relational structure of the original dataset rather than storing everything as one large merged table.

Loading order is important because foreign-key relationships exist between the tables. Parent tables must be loaded before tables that reference them.

### Planned loading order

1. `customers`
2. `products`
3. `sellers`
4. `orders`
5. `order_items`
6. `payments`
7. `reviews`

After loading, we will validate:

- row counts
- primary-key uniqueness
- foreign-key relationships
- successful data insertion

In [21]:
from dotenv import load_dotenv
import os
import psycopg

load_dotenv()

database_url = os.getenv("DATABASE_URL")

In [22]:
database_url is not None

True